# **Knowledge Engineering: Autonomous Self-Driving Trailer Agent**

**Course:** ICS 3202: Artificial Intelligence — Practical Exercise 4  
**Group Members:**
1. Bill Maina 
2. Carl Mogasi 
3. Rooney Segera
---

**Step 1: Identify the Task**

The task is to develop a Knowledge-Based Agent (KBA) that helps an autonomous heavy-cargo trailer decide optimal, safe routes and driving actions along the Kenyan highway network from **Nairobi to Kisumu** (extending the directed graph and Uniform Cost Search tree from Practical Exercise 3) based on road distances, terrain, cargo weight, fuel levels, weather conditions, and traffic incidents.

**Goal:** Make safe, fuel-feasible navigation and routing decisions with minimal risk of jackknifing, delays, or stranding.

**Step 2: Assemble Relevant Knowledge**

The system needs to know:
* **Static Highway Network Knowledge (from Practical Exercise 3):** Directed road edges, exact distances (km), speed limits (km/h), and terrain types across the 13 Kenyan towns (`Nairobi`, `Thika`, `Nyeri`, `Nyahururu`, `Naivasha`, `Narok`, `Bomet`, `Kisii`, `Nakuru`, `Kericho`, `Eldoret`, `Kakamega`, `Kisumu`).
* **Trailer Telemetry State:** Current cargo weight (tons), available fuel (liters), and fuel consumption rate (km/L).
* **Dynamic Environmental Percepts:** Real-time weather conditions (`Clear`, `Heavy_Rain`, `Dense_Fog`) and road incidents (`None`, `Accident_Congestion`, `Road_Closed`).
* **Domain Rules & Actions:** How weather, terrain, cargo load, and incidents affect road traversability, safe operating speed, and rerouting.

### API / External Data Declaration
* **OpenWeatherMap API (Optional Live Integration):** The `fetch_weather_api()` function in Step 4/5 queries OpenWeatherMap (`api.openweathermap.org/data/2.5/weather`) for Kenyan corridor nodes (e.g., `Naivasha,KE`, `Kericho,KE`) using the standard `requests` library to automatically inject real-time weather percepts into the Knowledge Base, with an automatic fallback to simulated corridor telemetry when offline or without an API key.

**Step 3: Define the Vocabulary**

* **Predicates (represented in Python as Knowledge Base keys, facts, and inference rules):**
  * `Road(origin, destination, distance_km, speed_limit, terrain)`
  * `Weather(origin, destination, condition)`
  * `Incident(origin, destination, status)`
  * `CargoLoad(tons)` and `FuelLevel(liters)`
  * `Traversable(origin, destination)`
  * `SafeSpeed(origin, destination, speed_kmh)`
  * `OptimalRoute(start, goal)`
* **Constants:**
  * **Locations:** `Nairobi`, `Naivasha`, `Nakuru`, `Kericho`, `Kisumu`, `Narok`, `Bomet`, `Kisii`, `Thika`, `Nyeri`, `Nyahururu`, `Eldoret`, `Kakamega`
  * **Terrain Types:** `Escarpment`, `Highway`, `Highland`, `Hilly`, `Slope`, `Plains`, `Lakeside`
  * **Weather Conditions:** `Clear`, `Heavy_Rain`, `Dense_Fog`
  * **Incidents:** `None`, `Accident_Congestion`, `Road_Closed`

**Step 4: Encode General Knowledge about the Domain** `[3 marks]`

Below we explicitly declare the domain vocabulary, the Practical Exercise 3 directed graph facts, and the general domain rules governing safe trailer navigation.

In [2]:
%pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.5.2-cp313-cp313-win_amd64.whl.metadata (47 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.5.2-cp313-cp313-win_amd64.whl (206 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   --------------------


[notice] A new release of pip is available: 25.1.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import heapq
import heapq
import json
import urllib.request
# 1. Explicit Domain Vocabulary
VOCABULARY = {
    'locations': [
        'Nairobi', 'Naivasha', 'Nakuru', 'Kericho', 'Kisumu',
        'Narok', 'Bomet', 'Kisii', 'Thika', 'Nyeri', 'Nyahururu', 'Eldoret', 'Kakamega'
    ],
    'weather_states': ['Clear', 'Heavy_Rain', 'Dense_Fog'],
    'incident_states': ['None', 'Accident_Congestion', 'Road_Closed'],
    'terrain_types': ['Escarpment', 'Highway', 'Highland', 'Hilly', 'Slope', 'Plains', 'Lakeside']
}

# 2. General Domain Rules (Condition -> Action / Speed Multiplier)
domain_rules = {
    'Road_Closed': {'traversable': False, 'speed_factor': 0.0, 'action': 'Prune route — road is impassable.'},
    'Escarpment_Heavy_Rain_Heavy_Cargo': {'traversable': False, 'speed_factor': 0.0, 'action': 'Prune route — high jackknife risk (>20t on wet escarpment).'},
    'Heavy_Rain': {'traversable': True, 'speed_factor': 0.65, 'action': 'Reduce speed by 35% and engage wet-traction control.'},
    'Dense_Fog': {'traversable': True, 'speed_factor': 0.65, 'action': 'Reduce speed by 35% and activate fog beacons.'},
    'Accident_Congestion': {'traversable': True, 'speed_factor': 0.50, 'action': 'Reduce speed by 50% due to traffic bottleneck.'},
    'Clear': {'traversable': True, 'speed_factor': 1.00, 'action': 'Proceed at normal posted highway speed limit.'}
}

# 3. Static Highway Knowledge Base (Exact Directed Graph & Distances from PE3)
knowledge_base = {
    'roads': {
        'Nairobi': {
            'Naivasha': {'distance_km': 93.6, 'speed_limit_kmh': 80, 'terrain': 'Escarpment'},
            'Narok': {'distance_km': 145.0, 'speed_limit_kmh': 80, 'terrain': 'Plains'},
            'Thika': {'distance_km': 42.0, 'speed_limit_kmh': 100, 'terrain': 'Highway'}
        },
        'Thika': {'Nyeri': {'distance_km': 108.0, 'speed_limit_kmh': 80, 'terrain': 'Highland'}},
        'Nyeri': {'Nyahururu': {'distance_km': 79.0, 'speed_limit_kmh': 70, 'terrain': 'Highland'}},
        'Nyahururu': {'Nakuru': {'distance_km': 108.0, 'speed_limit_kmh': 80, 'terrain': 'Escarpment'}},
        'Naivasha': {'Nakuru': {'distance_km': 69.7, 'speed_limit_kmh': 100, 'terrain': 'Highway'}},
        'Narok': {'Bomet': {'distance_km': 81.0, 'speed_limit_kmh': 80, 'terrain': 'Hilly'}},
        'Bomet': {'Kisii': {'distance_km': 82.4, 'speed_limit_kmh': 70, 'terrain': 'Hilly'}},
        'Kisii': {'Kisumu': {'distance_km': 107.0, 'speed_limit_kmh': 70, 'terrain': 'Lakeside'}},
        'Nakuru': {
            'Kericho': {'distance_km': 107.0, 'speed_limit_kmh': 80, 'terrain': 'Highland'},
            'Eldoret': {'distance_km': 171.0, 'speed_limit_kmh': 100, 'terrain': 'Highway'}
        },
        'Kericho': {'Kisumu': {'distance_km': 83.9, 'speed_limit_kmh': 80, 'terrain': 'Slope'}},
        'Eldoret': {'Kakamega': {'distance_km': 106.0, 'speed_limit_kmh': 80, 'terrain': 'Highway'}},
        'Kakamega': {'Kisumu': {'distance_km': 53.4, 'speed_limit_kmh': 80, 'terrain': 'Plains'}},
        'Kisumu': {}
    },
    'weather': {},
    'incidents': {},
    'trailer_state': {'cargo_tons': 28, 'fuel_liters': 180, 'km_per_liter': 2.5}
}

print("General domain rules and PE3 highway graph encoded successfully.")

General domain rules and PE3 highway graph encoded successfully.


**Step 5: Add New Facts to the Knowledge Base** `[4 marks]`

Here we define functions to view the Knowledge Base (`show_kb()`), add facts programmatically or via API (`add_new_weather_fact()`, `add_new_incident_fact()`, `fetch_weather_api()`), and add new facts interactively via user input (`add_fact_interactive()`).

In [4]:
def show_kb():
    print("\n--- Current Dynamic Knowledge Base ---")
    print("Trailer State:", knowledge_base['trailer_state'])
    print("Weather Facts:")
    if not knowledge_base['weather']:
        print("  (All corridors Clear)")
    for edge, cond in knowledge_base['weather'].items():
        print(f"  {edge[0]} -> {edge[1]:10} ➜ {cond}")
    print("Incident Facts:")
    if not knowledge_base['incidents']:
        print("  (No active incidents)")
    for edge, inc in knowledge_base['incidents'].items():
        print(f"  {edge[0]} -> {edge[1]:10} ➜ {inc}")
    print("--------------------------------------")

def add_new_weather_fact(origin, destination, condition):
    """Add a new weather fact directly to the KB (without user input)."""
    knowledge_base['weather'][(origin.title(), destination.title())] = condition
    print(f"Added weather fact: '{origin.title()} -> {destination.title()}' ➜ '{condition}'")

def add_new_incident_fact(origin, destination, incident):
    """Add a new road incident fact directly to the KB (without user input)."""
    knowledge_base['incidents'][(origin.title(), destination.title())] = incident
    print(f"Added incident fact: '{origin.title()} -> {destination.title()}' ➜ '{incident}'")

def fetch_weather_api(city_name, origin, destination, api_key=None):
    """Optional API integration to pull live Kenyan weather into the KB using built-in urllib."""
    if api_key:
        try:
            url = f"http://api.openweathermap.org/data/2.5/weather?q={city_name},KE&appid={api_key}"
            with urllib.request.urlopen(url, timeout=5) as response:
                data = json.loads(response.read().decode("utf-8"))
            main = data.get("weather", [{}])[0].get("main", "").lower()
            condition = "Heavy_Rain" if "rain" in main else "Clear"
            add_new_weather_fact(origin, destination, condition)
            return
        except Exception as e:
            print(f"API unavailable ({e}), using simulated sensor telemetry.")
    add_new_weather_fact(origin, destination, "Heavy_Rain")
    
# Programmatic fact injection (simulating live weather/traffic feeds)
fetch_weather_api('Naivasha', 'Nairobi', 'Naivasha')
add_new_weather_fact('Kericho', 'Kisumu', 'Dense_Fog')
add_new_incident_fact('Nakuru', 'Kericho', 'Accident_Congestion')

def add_fact_interactive():
    """Add new knowledge (facts) to the KB interactively via user input."""
    choice = input("\nDo you want to report a new road incident? (yes/no): ").strip().lower()
    if choice == 'yes':
        orig = input("Enter origin town (e.g., Narok): ").strip().title()
        dest = input("Enter destination town (e.g., Bomet): ").strip().title()
        inc = input("Enter incident (Road_Closed / Accident_Congestion / Clear): ").strip()
        if inc.lower() == 'clear':
            knowledge_base['incidents'].pop((orig, dest), None)
            print(f"Cleared incidents on '{orig} -> {dest}'")
        else:
            add_new_incident_fact(orig, dest, inc)

add_fact_interactive()
show_kb()

Added weather fact: 'Nairobi -> Naivasha' ➜ 'Heavy_Rain'
Added weather fact: 'Kericho -> Kisumu' ➜ 'Dense_Fog'
Added incident fact: 'Nakuru -> Kericho' ➜ 'Accident_Congestion'
Added incident fact: 'Narok -> Nairobi' ➜ 'Accident_congestion'

--- Current Dynamic Knowledge Base ---
Trailer State: {'cargo_tons': 28, 'fuel_liters': 180, 'km_per_liter': 2.5}
Weather Facts:
  Nairobi -> Naivasha   ➜ Heavy_Rain
  Kericho -> Kisumu     ➜ Dense_Fog
Incident Facts:
  Nakuru -> Kericho    ➜ Accident_Congestion
  Narok -> Nairobi    ➜ Accident_congestion
--------------------------------------


**Steps 6 & 7: Encode Specific Problem Instance (Facts) & Pose Queries and Get Answers** `[3 marks]`

In Step 6, we specify the trailer's current mission instance (start city, destination city, and sensor percepts). In Step 7, the inference engine (`evaluate_segment()` and `trailer_route_advice()`) evaluates every road segment against the domain rules and runs Uniform Cost Search over the inferred safe segments.

In [5]:
# Step 6: Specific Problem Instance (Current Mission & Sensor Observation)
start_city = input("Enter start city (default Nairobi): ").strip().title() or "Nairobi"
goal_city = input("Enter destination city (default Kisumu): ").strip().title() or "Kisumu"

# Step 7: Inference Engine & Query Resolution
def evaluate_segment(u, v):
    """Deduces whether segment (u, v) is safe and computes its effective speed and travel time."""
    if u not in knowledge_base['roads'] or v not in knowledge_base['roads'][u]:
        return {'traversable': False, 'advice': f"Unknown road segment: {u} -> {v}"}

    road = knowledge_base['roads'][u][v]
    weather = knowledge_base['weather'].get((u, v), 'Clear')
    incident = knowledge_base['incidents'].get((u, v), 'None')
    cargo = knowledge_base['trailer_state']['cargo_tons']

    # Check unknown percepts safely
    if weather not in domain_rules:
        return {'traversable': False, 'advice': f"Unknown weather condition '{weather}' — halt and verify."}
    if incident != 'None' and incident not in domain_rules:
        return {'traversable': False, 'advice': f"Unknown incident '{incident}' — proceed with extreme caution."}

    # Rule 1: Road Closed
    if incident == 'Road_Closed':
        return {'traversable': False, 'advice': domain_rules['Road_Closed']['action']}

    # Rule 2: Escarpment + Heavy Rain + Heavy Cargo (>20t)
    if road['terrain'] == 'Escarpment' and weather == 'Heavy_Rain' and cargo > 20:
        return {'traversable': False, 'advice': domain_rules['Escarpment_Heavy_Rain_Heavy_Cargo']['action']}

    # Rules 3 & 4: Compute safe operating speed
    speed_factor = domain_rules[weather]['speed_factor']
    advice_notes = [domain_rules[weather]['action']]
    if incident in domain_rules:
        speed_factor *= domain_rules[incident]['speed_factor']
        advice_notes.append(domain_rules[incident]['action'])

    safe_speed = road['speed_limit_kmh'] * speed_factor
    travel_time = road['distance_km'] / safe_speed
    fuel_needed = road['distance_km'] / knowledge_base['trailer_state']['km_per_liter']

    return {
        'traversable': True,
        'distance_km': road['distance_km'],
        'safe_speed_kmh': round(safe_speed, 1),
        'travel_time_hrs': round(travel_time, 2),
        'fuel_needed_l': round(fuel_needed, 1),
        'advice': " | ".join(advice_notes)
    }

def trailer_route_advice(start, goal):
    """Queries the KB to find the safest, fastest route from start to goal."""
    if start not in knowledge_base['roads'] or goal not in knowledge_base['roads']:
        return f"Unknown location(s): '{start}' or '{goal}' not in highway network."

    frontier = [(0.0, start, [start], 0.0, 0.0)]  # (time_hrs, node, path, dist_km, fuel_l)
    explored = set()

    while frontier:
        curr_time, u, path, curr_dist, curr_fuel = heapq.heappop(frontier)
        if u == goal:
            avail_fuel = knowledge_base['trailer_state']['fuel_liters']
            fuel_msg = (
                f"Sufficient ({curr_fuel:.1f}L / {avail_fuel}L)"
                if curr_fuel <= avail_fuel else
                f"REFUEL ALERT: Needs {curr_fuel:.1f}L, Available {avail_fuel}L"
            )
            return (
                f"Optimal Safe Route: {' -> '.join(path)}\n"
                f"Estimated Time    : {curr_time:.2f} hrs\n"
                f"Total Distance    : {curr_dist:.1f} km\n"
                f"Fuel Status       : {fuel_msg}"
            )
        if u not in explored:
            explored.add(u)
            for v in knowledge_base['roads'].get(u, {}):
                status = evaluate_segment(u, v)
                if status['traversable']:
                    heapq.heappush(frontier, (
                        curr_time + status['travel_time_hrs'],
                        v,
                        path + [v],
                        curr_dist + status['distance_km'],
                        curr_fuel + status['fuel_needed_l']
                    ))
    return "No safe traversable route found under current road/weather constraints."

print("\n=== Autonomous Trailer Decision System ===")
print(f"Mission: {start_city} to {goal_city}")
print(trailer_route_advice(start_city, goal_city))


=== Autonomous Trailer Decision System ===
Mission: Nairobi to Narok
Optimal Safe Route: Nairobi -> Narok
Estimated Time    : 1.81 hrs
Total Distance    : 145.0 km
Fuel Status       : Sufficient (58.0L / 180L)


**Step 8: Querying the Knowledge Base**

We pose targeted queries to verify how the agent evaluates specific segments and sub-routes under known and unknown conditions.

In [6]:
# Query individual highway segments and alternative sub-routes
print("Segment Advice (Nairobi -> Naivasha):", evaluate_segment('Nairobi', 'Naivasha'))
print("Segment Advice (Nairobi -> Narok)   :", evaluate_segment('Nairobi', 'Narok'))
print("Segment Advice (Kericho -> Kisumu)  :", evaluate_segment('Kericho', 'Kisumu'))
print("\nSub-route Query (Nakuru -> Kisumu):")
print(trailer_route_advice('Nakuru', 'Kisumu'))

Segment Advice (Nairobi -> Naivasha): {'traversable': False, 'advice': 'Prune route — high jackknife risk (>20t on wet escarpment).'}
Segment Advice (Nairobi -> Narok)   : {'traversable': True, 'distance_km': 145.0, 'safe_speed_kmh': 80.0, 'travel_time_hrs': 1.81, 'fuel_needed_l': 58.0, 'advice': 'Proceed at normal posted highway speed limit.'}
Segment Advice (Kericho -> Kisumu)  : {'traversable': True, 'distance_km': 83.9, 'safe_speed_kmh': 52.0, 'travel_time_hrs': 1.61, 'fuel_needed_l': 33.6, 'advice': 'Reduce speed by 35% and activate fog beacons.'}

Sub-route Query (Nakuru -> Kisumu):
Optimal Safe Route: Nakuru -> Eldoret -> Kakamega -> Kisumu
Estimated Time    : 3.70 hrs
Total Distance    : 330.4 km
Fuel Status       : Sufficient (132.2L / 180L)


**Step 9: Debugging the Knowledge Base** `[2 marks]`

We test valid, edge-case, and faulty/unknown percepts and locations to verify graceful handling and rule enforcement.

In [7]:
print("\n--- Debugging the Knowledge Base ---")

# 1. Test faulty/unknown weather percepts on a segment
test_weather_signals = ['Clear', 'Heavy_Rain', 'Dense_Fog', 'Hailstorm', 'Sandstorm', 'None']
for signal in test_weather_signals:
    knowledge_base['weather'][('Nairobi', 'Thika')] = signal
    res = evaluate_segment('Nairobi', 'Thika')
    print(f"Weather: {signal:12} -> Traversable: {str(res['traversable']):5} | Advice: {res['advice']}")

# Restore Nairobi -> Thika to Clear
knowledge_base['weather'][('Nairobi', 'Thika')] = 'Clear'

# 2. Test unknown city query
print("\nUnknown City Query (Nairobi -> Mombasa):")
print(trailer_route_advice('Nairobi', 'Mombasa'))


--- Debugging the Knowledge Base ---
Weather: Clear        -> Traversable: True  | Advice: Proceed at normal posted highway speed limit.
Weather: Heavy_Rain   -> Traversable: True  | Advice: Reduce speed by 35% and engage wet-traction control.
Weather: Dense_Fog    -> Traversable: True  | Advice: Reduce speed by 35% and activate fog beacons.
Weather: Hailstorm    -> Traversable: False | Advice: Unknown weather condition 'Hailstorm' — halt and verify.
Weather: Sandstorm    -> Traversable: False | Advice: Unknown weather condition 'Sandstorm' — halt and verify.
Weather: None         -> Traversable: False | Advice: Unknown weather condition 'None' — halt and verify.

Unknown City Query (Nairobi -> Mombasa):
Unknown location(s): 'Nairobi' or 'Mombasa' not in highway network.


**Step 10: Final Reflection (Challenges & Recommended Improvements)** `[3 marks]`

### Challenges Encountered
1. **Translating Numeric Graph Search into Symbolic Knowledge:** In Practical Exercise 3, Uniform Cost Search only minimized static distances (`354.2 km` via Naivasha–Nakuru–Kericho). Introducing symbolic weather, terrain, and cargo constraints required decoupling static topology from dynamic environmental percepts so hazardous segments (like the wet Naivasha escarpment) are pruned without mutating the underlying map.
2. **Handling Incomplete or Unknown Percepts:** Real-world telemetry can produce unmodeled weather or incident strings. Explicit validation against `domain_rules` was necessary so the agent defaults to a safe halt/caution state rather than crashing.

### Recommended Improvements
1. **Dynamic Refueling Waypoint Planning:** Currently, the agent alerts when total fuel required exceeds the trailer's `180L` tank. Integrating fuel station nodes along the highway graph would allow the inference engine to automatically insert a refueling stop when fuel is insufficient.
2. **Automated Live Traffic & Weather Polling:** Connecting live KenHA (Kenya National Highways Authority) road closure feeds alongside continuous OpenWeatherMap API polling would allow autonomous mid-journey rerouting without manual `TELL` inputs.

---


